<a href="https://colab.research.google.com/github/yuhui-0611/SEED/blob/main/SEED_YB_%EC%99%B8%EC%8B%9D%EC%97%85.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 설치
!pip install linearmodels openpyxl scikit-learn

# 라이브러리 불러오기
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
from linearmodels.panel import PanelOLS, PooledOLS, RandomEffects, compare
from google.colab import files

In [ ]:
from google.colab import files
uploaded = files.upload()

df = pd.read_excel("외식업_통합.xlsx")

Saving 외식업_통합.xlsx to 외식업_통합 (1).xlsx


In [ ]:
# 연도·분기 분리 후 datetime 인덱스 생성
df[['연도', '분기']] = df['연도_분기'].str.extract(r'(\d{4})-Q(\d)')
df['연도'] = df['연도'].astype(int)
df['분기'] = df['분기'].astype(int)
df['연도_분기_dt'] = pd.to_datetime(df['연도'].astype(str) + 'Q' + df['분기'].astype(str))

# 파생변수 생성
df['log_환산임대료'] = np.log(df['환산임대료'])
df['프랜차이즈비율'] = df['프랜차이즈 점포수'] / df['전체점포수']  # 또는 프랜차이즈 + 일반 점포수

# 분석 변수 목록 정의
indep_vars = [
    '프랜차이즈비율', '개업률', '1년 생존율 차분', '5년 생존율 차분',
    'log인구', 'log_환산임대료', '10년영업기간', '30년영업기간',
    '실업률', '소비자심리지수(평균)', '소득분위'
]

# 결측치 제거 및 필요한 변수만 추출
columns = ['행정구역', '연도_분기_dt', '폐업률'] + indep_vars
df_model = df[columns].dropna()

# 표준화 (z-score)
scaler = StandardScaler()
df_model_std = df_model.copy()
df_model_std[[f"{col}_표준화" for col in indep_vars]] = scaler.fit_transform(df_model[indep_vars])

# 패널 인덱스 설정 (자치구 + 시점)
df_panel = df_model_std.set_index(['행정구역', '연도_분기_dt'])

# 종속변수
y = df_panel['폐업률']

# 독립변수 (표준화된 변수만 사용)
X_vars_std = [f"{col}_표준화" for col in indep_vars]
X = df_panel[X_vars_std].copy()
X["const"] = 1  # 상수항 추가


<ipython-input-15-55f72e35b76b>:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['연도_분기_dt'] = pd.to_datetime(df['연도'].astype(str) + 'Q' + df['분기'].astype(str))


# **F-검정 (poolability test)**

In [ ]:
# 모형 불러오기
from linearmodels.panel import PooledOLS, PanelOLS, compare

# Pooled OLS 모형 추정
mod_pool = PooledOLS(y, X)
res_pool = mod_pool.fit()

# Fixed Effects 모형 추정
mod_fe = PanelOLS(y, X, entity_effects=True)
res_fe = mod_fe.fit()

# Poolability Test 수행
pool_test = compare({"Pooled OLS": res_pool, "Fixed Effects": res_fe})

# 결과 요약 출력
print("◆ Pooled OLS 결과\n")
print(res_pool.summary)

print("\n\n◆ Fixed Effects 결과\n")
print(res_fe.summary)

# Poolability F-test 직접 출력
f_test_stat = res_fe.f_statistic.stat
f_test_pval = res_fe.f_statistic.pval

print("\n\n📊 Poolability F-test")
print(f"F-statistic: {f_test_stat:.4f}")
print(f"p-value: {f_test_pval:.4f}")

# 해석 메시지 출력
if f_test_pval < 0.05:
    print("✅ 고정효과 모형이 더 적합합니다 (Pooled OLS 기각).")
else:
    print("✅ Pooled OLS로도 충분합니다 (고정효과 불필요).")


◆ Pooled OLS 결과

                          PooledOLS Estimation Summary                          
Dep. Variable:                    폐업률   R-squared:                        0.4460
Estimator:                  PooledOLS   R-squared (Between):              0.6914
No. Observations:                1040   R-squared (Within):               0.3828
Date:                Mon, May 19 2025   R-squared (Overall):              0.4460
Time:                        17:57:34   Log-likelihood                   -950.80
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      75.235
Entities:                          26   P-value                           0.0000
Avg Obs:                       40.000   Distribution:                 F(11,1028)
Min Obs:                       40.000                                           
Max Obs:                       40.000   F-statistic (robust):             75.235
           

# **Hausman 검정**

In [ ]:
from linearmodels.panel import RandomEffects, compare

# 1. 확률효과 모형 추정
mod_re = RandomEffects(y, X)
res_re = mod_re.fit()

# 2. 고정효과 모형은 기존 res_fe 사용

# 3. Hausman 검정: 두 모형 비교
print("📊 Hausman Test (Fixed Effects vs Random Effects)\n")
comparison = compare({"Fixed Effects": res_fe, "Random Effects": res_re})
print(comparison)

📊 Hausman Test (Fixed Effects vs Random Effects)

                    Model Comparison                    
                         Fixed Effects    Random Effects
--------------------------------------------------------
Dep. Variable                      폐업률               폐업률
Estimator                     PanelOLS     RandomEffects
No. Observations                  1040              1040
Cov. Est.                   Unadjusted        Unadjusted
R-squared                       0.4936            0.4358
R-Squared (Within)              0.4936            0.3995
R-Squared (Between)            -45.778            0.6175
R-Squared (Overall)            -8.9804            0.4442
F-statistic                     88.889            72.182
P-value (F-stat)                0.0000            0.0000
=====================     ============   ===============
프랜차이즈비율_표준화                    -0.4936           -0.1487
                             (-9.2716)         (-6.5492)
개업률_표준화                         0.2407

p-value

In [ ]:
import numpy as np
from scipy import stats

# 1. 상수항 제거 (공분산 역행렬 계산 안정성 확보를 위함)
b_FE = res_fe.params.drop('const')
b_RE = res_re.params.drop('const')

cov_FE = res_fe.cov.drop(index='const', columns='const')
cov_RE = res_re.cov.drop(index='const', columns='const')

# 2. Hausman 통계량 계산
b_diff = b_FE - b_RE
cov_diff = cov_FE - cov_RE

# 3. 정규 형태의 검정 통계량 계산
try:
    stat = np.dot(np.dot(b_diff.T, np.linalg.inv(cov_diff)), b_diff)
    df = len(b_diff)
    p_value = 1 - stats.chi2.cdf(stat, df)

    print(f"Hausman Test Statistic: {stat:.4f}")
    print(f"Degrees of Freedom: {df}")
    print(f"P-Value: {p_value:.4f}")

    if p_value < 0.05:
        print("✅ 고정효과(FE) 모형이 더 적합합니다 (RE 기각).")
    else:
        print("✅ 확률효과(RE) 모형을 사용할 수 있습니다 (귀무가설 채택).")
except np.linalg.LinAlgError:
    print("❌ 공분산 행렬 차이가 singular하여 역행렬을 계산할 수 없습니다.")
    print("   → 계수 수가 많거나 다중공선성 문제가 있을 수 있습니다.")


Hausman Test Statistic: 396.5950
Degrees of Freedom: 11
P-Value: 0.0000
✅ 고정효과(FE) 모형이 더 적합합니다 (RE 기각).


**Hausman Test 해석**
- 두 모형 모두 통계적으로 유의미하지만, FE 모형의 설명력 (R-squared Within = 0.4936) 이 RE 모형 (0.3995) 보다 높다
- 프랜차이즈비율_표준화의 계수는 FE에서 -0.4936, RE에서 -0.1487로 상당한 차이를 보였다
- log인구_표준화 변수는 FE에서 2.7096, RE에서는 0.0659로 계수 차이가 극단적이다
- 이는 확률효과 모형이 개체(자치구)의 고정된 특성을 충분히 설명하지 못하고 있다는 뜻
- 두번째 코드에서 Hausman 통계량이 396.5950으로 매우 큰 값을 가진다
- 즉, FE와 RE 모형의 계수 차이가 통계적으로 매우 크다
- 이는 귀무가설인 “RE가 더 효율적이며 일관되다”는 주장을 기각
- 고정효과(FE) 모형이 더 일관되고 적절

# **1계 자기상관 검정**

In [ ]:
# 고정효과 모델 추정
mod_fe = PanelOLS(y, X, entity_effects=True)
res_fe = mod_fe.fit()


# 1. FE 모델 잔차 추출 (res_fe는 FE 회귀 결과)
residuals = res_fe.resids.copy()

# 2. 1기 시차 생성 (자치구별)
residuals_shifted = residuals.groupby(level=0).shift(1)

# 3. NaN 제거 (시차가 없는 첫 시점 제거)
valid_idx = ~residuals_shifted.isna()
y_ac = residuals[valid_idx]
X_ac = sm.add_constant(residuals_shifted[valid_idx])

# 4. 회귀로 자기상관 계수 추정
import statsmodels.api as sm
model_ac = sm.OLS(y_ac, X_ac).fit()
rho = model_ac.params[1]  # 자기상관 계수

# 5. t-통계량 및 p-value 계산
from scipy import stats
n_obs = len(y_ac)
t_stat = rho / model_ac.bse[1]
p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=n_obs - 2))

# 6. 결과 출력
print(f"📌 1계 자기상관 계수 (rho): {rho:.4f}")
print(f"t-통계량: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

📌 1계 자기상관 계수 (rho): 0.0845
t-통계량: 2.6825
p-value: 0.0074


<ipython-input-25-4b41481b4586>:20: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  rho = model_ac.params[1]  # 자기상관 계수
<ipython-input-25-4b41481b4586>:25: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  t_stat = rho / model_ac.bse[1]


**1계 자기상관 검정 해석**
- rho는 0.1 미만으로 작은 양의 자기상관 존재
- 즉, 고정효과 회귀모형의 잔차가 이전 시점의 잔차와 약간 양의 방향으로 연관되어 있음을 의미
- t-통계량 = 2.68, p-value = 0.0074
- 귀무가설(자기상관 없음)을 기각
- 즉, 잔차에 통계적으로 유의미한 1차 자기상관이 존재

In [ ]:
# FE 모델 잔차 불러오기
residuals = res_fe.resids.copy()

# 독립변수와 잔차 간 상관계수 계산
corr_with_resid = df_panel[X_vars_std].corrwith(residuals)

# 결과 출력
print("📌 잔차와 변수 간 상관관계 (상위 변수):")
print(corr_with_resid.sort_values(key=abs, ascending=False))


📌 잔차와 변수 간 상관관계 (상위 변수):
실업률_표준화            1.075245e-15
10년영업기간_표준화        4.579747e-16
log인구_표준화          2.986792e-16
프랜차이즈비율_표준화       -2.654926e-16
5년 생존율 차분_표준화      2.165424e-16
소득분위_표준화           1.991194e-16
30년영업기간_표준화        1.659329e-16
1년 생존율 차분_표준화      1.393836e-16
개업률_표준화           -1.294276e-16
log_환산임대료_표준화     -9.955972e-17
소비자심리지수(평균)_표준화    9.790039e-17
dtype: float64


In [ ]:
# 1. 시차 변수 생성 (자치구별로 1기 시차)
lag_vars = ['실업률_표준화', '소비자심리지수(평균)_표준화']
for var in lag_vars:
    df_panel[f'{var}_lag1'] = df_panel.groupby(level=0)[var].shift(1)

# 2. 독립변수 목록 구성
X_base = [col for col in X.columns if col != 'const']
X_lag = [f"{var}_lag1" for var in lag_vars]
X_all = X_base + X_lag

# 3. 분석용 데이터 구성 및 결측치 제거
df_model_lag = df_panel[X_all + ['폐업률']].dropna()
y_lag = df_model_lag['폐업률']
X_lag_df = df_model_lag[X_all].copy()
X_lag_df["const"] = 1

# 4. 고정효과(FE) 모형 추정 (robust SE)
model_fe_lag = PanelOLS(y_lag, X_lag_df, entity_effects=True)
res_fe_lag = model_fe_lag.fit(cov_type="robust")

# 5. 자기상관 검정 (AR(1) 방식)
residuals = res_fe_lag.resids.copy()
resid_shifted = residuals.groupby(level=0).shift(1)

# NaN 제거
valid_idx = ~resid_shifted.isna()
y_ac = residuals[valid_idx]
X_ac = sm.add_constant(resid_shifted[valid_idx])

# 회귀 수행
model_ac = sm.OLS(y_ac, X_ac).fit()
rho = model_ac.params[1]  # 자기상관 계수

# t-통계량, p-value
from scipy import stats
n_obs = len(y_ac)
t_stat = rho / model_ac.bse[1]
p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=n_obs - 2))

# 6. 결과 출력
print(f"📌 시차 변수 포함 후 자기상관 계수 (rho): {rho:.4f}")
print(f"t-통계량: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

📌 시차 변수 포함 후 자기상관 계수 (rho): 0.0817
t-통계량: 2.5668
p-value: 0.0104


<ipython-input-31-93be75be3b0a>:32: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  rho = model_ac.params[1]  # 자기상관 계수
<ipython-input-31-93be75be3b0a>:37: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  t_stat = rho / model_ac.bse[1]


- 계수 자체는 0.0845 → 0.0817로 약간 감소했지만, 여전히 0.08 수준의 양의 자기상관이 남아 있다

**lag2 추가 적용**

lag1과 2를 동시에 적용하는 이유

: 현실에서는 직전 시점과 그보다 이전의 정보가 동시에 영향을 미치기에 lag2만 넣게 되면 계수 왜곡이 발생함

In [ ]:
# 1. lag1, lag2 생성 (자치구 기준 시차)
lag_vars = ['실업률_표준화', '소비자심리지수(평균)_표준화']
for var in lag_vars:
    df_panel[f'{var}_lag1'] = df_panel.groupby(level=0)[var].shift(1)
    df_panel[f'{var}_lag2'] = df_panel.groupby(level=0)[var].shift(2)

# 2. 전체 독립변수 목록 재구성
X_base = [col for col in X.columns if col != 'const']  # 기존 변수
X_lags = [f"{var}_lag1" for var in lag_vars] + [f"{var}_lag2" for var in lag_vars]  # lag1 + lag2
X_all = X_base + X_lags

# 3. 결측치 제거 및 데이터 준비
df_model_lag = df_panel[X_all + ['폐업률']].dropna()
y_lag = df_model_lag['폐업률']
X_lag_df = df_model_lag[X_all].copy()
X_lag_df['const'] = 1

# 4. FE 모형 추정 (robust SE 적용)
model_fe_lag = PanelOLS(y_lag, X_lag_df, entity_effects=True)
res_fe_lag = model_fe_lag.fit(cov_type="robust")

# 5. 자기상관 검정 (잔차 vs 1기 시차 잔차 회귀)
residuals = res_fe_lag.resids.copy()
resid_shifted = residuals.groupby(level=0).shift(1)

# NaN 제거
valid_idx = ~resid_shifted.isna()
y_ac = residuals[valid_idx]
X_ac = sm.add_constant(resid_shifted[valid_idx])

# 자기상관 계수 회귀
model_ac = sm.OLS(y_ac, X_ac).fit()
rho = model_ac.params.iloc[1]
t_stat = rho / model_ac.bse.iloc[1]
n_obs = len(y_ac)
p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=n_obs - 2))

# 6. 출력
print(res_fe_lag.summary)
print("\n📌 시차 변수(lag1, lag2) 포함 후 자기상관 검정 결과:")
print(f"자기상관 계수 (rho): {rho:.4f}")
print(f"t-통계량: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

                          PanelOLS Estimation Summary                           
Dep. Variable:                    폐업률   R-squared:                        0.4740
Estimator:                   PanelOLS   R-squared (Between):             -51.419
No. Observations:                 988   R-squared (Within):               0.4740
Date:                Mon, May 19 2025   R-squared (Overall):             -10.100
Time:                        19:27:43   Log-likelihood                   -748.19
Cov. Estimator:                Robust                                           
                                        F-statistic:                      56.888
Entities:                          26   P-value                           0.0000
Avg Obs:                       38.000   Distribution:                  F(15,947)
Min Obs:                       38.000                                           
Max Obs:                       38.000   F-statistic (robust):             57.768
                            

- 여전히 자기상관 존재
- 고정효과 모형 추정 시 자치구 단위로 클러스터된 표준오차를 적용하여 이분산성과 동질적 자기상관 문제 보정
- 클러스터 표준오차는 단위별 시간적 종속성을 반영하기에 계수 추정 유의성 해석에 안정적 기준 제공

# **Clustered SE 적용 고정효과모형**

In [ ]:
from linearmodels.panel import PanelOLS

# 1. 기존 lag1 + lag2 포함된 독립변수 목록 사용
X_all = X_base + [f"{var}_lag1" for var in lag_vars] + [f"{var}_lag2" for var in lag_vars]

# 2. 분석 데이터 구성
df_model_cluster = df_panel[X_all + ['폐업률']].dropna()
y_cluster = df_model_cluster['폐업률']
X_cluster = df_model_cluster[X_all].copy()
X_cluster["const"] = 1

# 3. 고정효과 회귀모형 (Clustered SE)
model_fe_cluster = PanelOLS(y_cluster, X_cluster, entity_effects=True)
res_fe_cluster = model_fe_cluster.fit(cov_type="clustered", cluster_entity=True)

# 4. 결과 출력
print(res_fe_cluster.summary)


                          PanelOLS Estimation Summary                           
Dep. Variable:                    폐업률   R-squared:                        0.4740
Estimator:                   PanelOLS   R-squared (Between):             -51.419
No. Observations:                 988   R-squared (Within):               0.4740
Date:                Mon, May 19 2025   R-squared (Overall):             -10.100
Time:                        19:40:43   Log-likelihood                   -748.19
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      56.888
Entities:                          26   P-value                           0.0000
Avg Obs:                       38.000   Distribution:                  F(15,947)
Min Obs:                       38.000                                           
Max Obs:                       38.000   F-statistic (robust):             379.24
                            

**클러스터 적용 이후의 자기상관 재검정**

In [ ]:
import statsmodels.api as sm
from scipy import stats

# 1. 잔차 추출 (res_fe_cluster = FE 모형 with clustered SE)
residuals = res_fe_cluster.resids.copy()

# 2. 1기 시차 생성 (자치구 기준)
resid_shifted = residuals.groupby(level=0).shift(1)

# 3. NaN 제거
valid_idx = ~resid_shifted.isna()
y_ac = residuals[valid_idx]
X_ac = sm.add_constant(resid_shifted[valid_idx])

# 4. OLS 회귀로 자기상관 계수 추정
model_ac = sm.OLS(y_ac, X_ac).fit()
rho = model_ac.params.iloc[1]
t_stat = rho / model_ac.bse.iloc[1]
n_obs = len(y_ac)
p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=n_obs - 2))

# 5. 출력
print("📌 클러스터 SE 적용 후 잔차 자기상관 검정 결과:")
print(f"자기상관 계수 (rho): {rho:.4f}")
print(f"t-통계량: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")


📌 클러스터 SE 적용 후 잔차 자기상관 검정 결과:
자기상관 계수 (rho): 0.0845
t-통계량: 2.6261
p-value: 0.0088


- 자기상관이 존재할 경우, 일반 표준오차 기반의 고정효과 회귀모형은 표준오차와 유의확률(p-value)이 왜곡될 수 있다
- 본 분석에서는 클러스터 표준오차(clustered standard errors)를 자치구 단위로 적용하여, 이러한 자기상관의 영향을 표준오차 추정에 반영하였다
- 따라서 잔차에서 자기상관이 여전히 유의하게 나타났더라도, 회귀계수의 추정과 유의성 해석은 통계적으로 신뢰 가능하다

# **이분산성 검정**

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan
import statsmodels.api as sm

# 1. 데이터 정리: 종속변수 + 독립변수(표준화)만 포함하여 결측치 제거
bp_data = df_panel[X_vars_std + ['폐업률']].dropna()

# 2. 종속변수와 독립변수 구분
y_ols = bp_data['폐업률']
X_ols = bp_data[X_vars_std]
X_ols = sm.add_constant(X_ols)  # 상수항 추가

# 3. OLS 모델 적합 (FE 아님)
ols_model = sm.OLS(y_ols, X_ols).fit()

# 4. 잔차 & 독립변수 추출
resid_bp = ols_model.resid
exog_bp = X_ols

# 5. Breusch-Pagan Test 수행
bp_test = het_breuschpagan(resid_bp, exog_bp)

# 6. 결과 출력
print("📌 Breusch-Pagan Test 결과:")
print(f"LM Statistic: {bp_test[0]:.4f}")
print(f"p-value: {bp_test[1]:.4f}")

# 7. 해석
if bp_test[1] < 0.05:
    print("✅ 이분산성이 존재합니다 (귀무가설 기각)")
else:
    print("✅ 이분산성이 없습니다 (귀무가설 채택)")


📌 Breusch-Pagan Test 결과:
LM Statistic: 70.9306
p-value: 0.0000
✅ 이분산성이 존재합니다 (귀무가설 기각)


**이분산성 검정 해석**
- p-value < 0.05이므로, 이분산성을 갖는다는 근거가 통계적으로 유의
- 해당 분석은 cluster 적용 이전의 자료를 이용한 것으로, 이미 자기상관이 검출되어 cluster 보정한 상태이기에 cluster se 유지하면 된다

# **고정효과 회귀모형**

In [ ]:
from linearmodels.panel import PanelOLS

# 1. 최종 데이터에서 종속변수와 독립변수 추출
y_final = df_panel['폐업률']
X_final = df_panel[X_vars_std].copy()
X_final["const"] = 1  # 상수항 추가

# 2. 고정효과(Fixed Effects) 모델 지정
mod_fe_final = PanelOLS(y_final, X_final, entity_effects=True)

# 3. 클러스터 표준오차 적용하여 추정 (자치구 단위 클러스터링)
res_fe_final = mod_fe_final.fit(cov_type='clustered', cluster_entity=True)

# 4. 결과 출력
print(res_fe_final.summary)


                          PanelOLS Estimation Summary                           
Dep. Variable:                    폐업률   R-squared:                        0.4936
Estimator:                   PanelOLS   R-squared (Between):             -45.778
No. Observations:                1040   R-squared (Within):               0.4936
Date:                Mon, May 19 2025   R-squared (Overall):             -8.9804
Time:                        20:08:52   Log-likelihood                   -784.91
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      88.889
Entities:                          26   P-value                           0.0000
Avg Obs:                       40.000   Distribution:                 F(11,1003)
Min Obs:                       40.000                                           
Max Obs:                       40.000   F-statistic (robust):             469.58
                            